In [ ]:
countdata_LMC <- read.csv('normalised_LMC_countdata.csv')

In [ ]:
countdata_MEF <- read.csv('normalised_MEF_countdata.csv')

In [ ]:
head(countdata_MEF)

In [ ]:
head(countdata_LMC)

In [ ]:
############################
## Load packages
############################
suppressPackageStartupMessages({
  library(pheatmap)
  library(dplyr)
  library(gridExtra)
  library(grid)
})

############################
## Helper: prepare MEF matrix from countdata_MEF
############################
prep_MEF_matrix <- function(countdata_MEF) {
  MEF_df <- as.data.frame(countdata_MEF)
  gene_col <- colnames(MEF_df)[1]
  gene_names <- toupper(MEF_df[[gene_col]])
  MEF_df[[gene_col]] <- NULL
  MEF_mat <- as.matrix(MEF_df)
  rownames(MEF_mat) <- gene_names
  MEF_mat
}

############################
## Helper: prepare LMC matrix from countdata_LMC
############################
prep_LMC_matrix <- function(countdata_LMC) {
  LMC_df <- as.data.frame(countdata_LMC)
  if (!"Gene_name" %in% colnames(LMC_df)) {
    stop("countdata_LMC must contain a 'Gene_name' column.")
  }
  gene_names <- toupper(LMC_df$Gene_name)
  LMC_df$Gene_name <- NULL
  if ("X" %in% colnames(LMC_df)) {
    LMC_df$X <- NULL
  }
  LMC_mat <- as.matrix(LMC_df)
  rownames(LMC_mat) <- gene_names
  LMC_mat
}

############################
## Collapse replicates to 4 groups
############################
collapse_to_4groups <- function(mat, groups) {
  stopifnot(all(unlist(groups) %in% colnames(mat)))
  collapsed <- data.frame(
    WT_Control    = rowMeans(mat[, groups$WT_Control,    drop = FALSE], na.rm = TRUE),
    WT_LPS        = rowMeans(mat[, groups$WT_LPS,        drop = FALSE], na.rm = TRUE),
    Sp3KO_Control = rowMeans(mat[, groups$Sp3KO_Control, drop = FALSE], na.rm = TRUE),
    Sp3KO_LPS     = rowMeans(mat[, groups$Sp3KO_LPS,     drop = FALSE], na.rm = TRUE)
  )
  rownames(collapsed) <- rownames(mat)
  collapsed
}

############################
## Safe row-scaling (z-score per gene)
############################
safe_row_scale <- function(mat) {
  t(apply(mat, 1, function(x) {
    mu <- mean(x, na.rm = TRUE)
    sdv <- sd(x, na.rm = TRUE)
    if (is.na(sdv) || sdv == 0) rep(0, length(x)) else (x - mu) / sdv
  }))
}

############################
## Annotation + colors
############################
make_annotation <- function() {
  annotation_col <- data.frame(
    Genotype  = c("WT","WT","Sp3KO","Sp3KO"),
    Condition = c("Control","LPS","Control","LPS"),
    stringsAsFactors = FALSE
  )
  rownames(annotation_col) <- c("WT_Control","WT_LPS","Sp3KO_Control","Sp3KO_LPS")

  ann_colors <- list(
    Genotype  = c(WT = "#66c2a5", Sp3KO = "#fc8d62"),
    Condition = c(Control = "#8da0cb", LPS = "#e78ac3")
  )
  list(annotation_col = annotation_col, ann_colors = ann_colors)
}

############################
## Heatmap color palette
############################
hm_colors <- colorRampPalette(
  c("#313695", "#74add1", "#ffffbf", "#f46d43", "#a50026")
)(200)

############################
## Align rows between MEF and LMC
############################
subset_and_align_rows <- function(mat_list, genes_upper) {
  sub_list <- lapply(mat_list, function(m) {
    keep <- intersect(genes_upper, rownames(m))
    m_sub <- m[keep, , drop = FALSE]
    m_sub <- m_sub[match(genes_upper, rownames(m_sub), nomatch = 0), , drop = FALSE]
    m_sub
  })
  common_genes <- Reduce(intersect, lapply(sub_list, rownames))
  if (length(common_genes) == 0) stop("No overlapping genes between MEF and LMC.")
  ordered_common <- genes_upper[genes_upper %in% common_genes]
  aligned <- lapply(sub_list, function(m) m[ordered_common, , drop = FALSE])
  aligned
}

############################
## pheatmap wrapper (no row clustering, scaled)
############################
make_heatmap_grob <- function(mat, title_text, ann) {
  desired_order <- c("WT_Control","WT_LPS","Sp3KO_Control","Sp3KO_LPS")
  mat <- mat[, desired_order, drop = FALSE]
  mat_scaled <- safe_row_scale(mat)   # <-- scaling applied here
  p <- pheatmap(
    mat_scaled,
    cluster_rows = FALSE,
    cluster_cols = FALSE,
    scale = "none",     # already manually scaled
    color = hm_colors,
    annotation_col = ann$annotation_col,
    annotation_colors = ann$ann_colors,
    show_rownames = TRUE,
    show_colnames = TRUE,
    fontsize_row = 7,
    fontsize_col = 9,
    angle_col = 45,
    cellheight = 10,
    cellwidth  = 20,
    border_color = NA,
    main = title_text,
    treeheight_row = 0,
    treeheight_col = 0,
    silent = TRUE
  )
  p$gtable
}

############################
## Gene list (uppercase)
############################
genes_of_interest <- c(
  "HOXA1","HOXA10","HOXA11","HOXA11OS","HOXA2","HOXA3","HOXA4","HOXA5","HOXA6","HOXA7","HOXA9",
  "HOXB1","HOXB2","HOXB3","HOXB3OS","HOXB4","HOXB5","HOXB5OS","HOXB6","HOXB7","HOXB8","HOXB9",
  "HOXC10","HOXC11","HOXC13","HOXC4","HOXC5","HOXC6","HOXC8","HOXC9",
  "HOXD10","HOXD11","HOXD3","HOXD3OS1","HOXD8","HOXD9",
  "PAX3","PAX6",
  "POU2F1","POU2F2","POU2F3",
  "POU3F1","POU3F2","POU3F3","POU3F4",
  "POU4F1","POU4F3","POU5F2","POU6F1"
)
genes_of_interest_upper <- toupper(genes_of_interest)

############################
## MAIN PIPELINE
############################

## --- MEF ---
MEF_mat <- prep_MEF_matrix(countdata_MEF)
MEF_groups <- list(
  WT_Control    = c("P7_4_Ctrl1","P7_4_Ctrl2","P8_4_Ctrl3","P8_4_Ctrl4","P8_4_Ctrl5"),
  WT_LPS        = c("P8_4_Lps3","P8_4_Lps5","P8_4_Lps4","P7_4_Lps1","P7_4_Lps2"),
  Sp3KO_Control = c("P4_5_Ctrl1","P4_5_Ctrl3","P4_5_Ctrl2"),
  Sp3KO_LPS     = c("P4_5_Lps2","P4_5_Lps3","P4_5_Lps1")
)
MEF_collapsed <- collapse_to_4groups(MEF_mat, MEF_groups)

## --- LMC ---
LMC_mat <- prep_LMC_matrix(countdata_LMC)
LMC_groups <- list(
  WT_Control    = c("X685.2_CTRL","X697.3_CTRL","X697.1_CTRL","X468.4_CTRL"),
  WT_LPS        = c("X468.4_LPS","X685.2_LPS","X697.3_LPS","X697.1_LPS"),
  Sp3KO_Control = c("X685.1_CTRL","X697.5_CTRL","X697.2_CTRL","X468.6_CTRL"),
  Sp3KO_LPS     = c("X697.5_LPS","X697.2_LPS","X685.1_LPS","X468.6_LPS")
)
LMC_collapsed <- collapse_to_4groups(LMC_mat, LMC_groups)

## --- Align genes ---
aligned_list <- subset_and_align_rows(
  list(MEF = MEF_collapsed, LMC = LMC_collapsed),
  genes_of_interest_upper
)
MEF_final <- aligned_list$MEF
LMC_final <- aligned_list$LMC

## --- Shared annotation ---
ann <- make_annotation()

## --- Create both heatmap grobs (scaled, not clustered) ---
grob_mef <- make_heatmap_grob(
  MEF_final,
  title_text = "MEF",
  ann = ann
)
grob_lmc <- make_heatmap_grob(
  LMC_final,
  title_text = "LMC",
  ann = ann
)

## --- Display side-by-side ---
grid.newpage()
grid.arrange(grob_mef, grob_lmc, ncol = 2)


In [ ]:
irf <- read.csv('irf_genes.csv')

In [ ]:
irf$X <- NULL

In [ ]:
irf

In [ ]:
############################
## Load packages
############################
suppressPackageStartupMessages({
  library(pheatmap)
  library(dplyr)
  library(gridExtra)
  library(grid)
})

############################
## Helper: prepare MEF matrix from countdata_MEF
############################
prep_MEF_matrix <- function(countdata_MEF) {
  MEF_df <- as.data.frame(countdata_MEF)
  gene_col <- colnames(MEF_df)[1]
  gene_names <- toupper(MEF_df[[gene_col]])
  MEF_df[[gene_col]] <- NULL
  MEF_mat <- as.matrix(MEF_df)
  rownames(MEF_mat) <- gene_names
  MEF_mat
}

############################
## Helper: prepare LMC matrix from countdata_LMC
############################
prep_LMC_matrix <- function(countdata_LMC) {
  LMC_df <- as.data.frame(countdata_LMC)
  if (!"Gene_name" %in% colnames(LMC_df)) {
    stop("countdata_LMC must contain a 'Gene_name' column.")
  }
  gene_names <- toupper(LMC_df$Gene_name)
  LMC_df$Gene_name <- NULL
  if ("X" %in% colnames(LMC_df)) {
    LMC_df$X <- NULL
  }
  LMC_mat <- as.matrix(LMC_df)
  rownames(LMC_mat) <- gene_names
  LMC_mat
}

############################
## Collapse replicates to 4 groups
############################
collapse_to_4groups <- function(mat, groups) {
  stopifnot(all(unlist(groups) %in% colnames(mat)))
  collapsed <- data.frame(
    WT_Control    = rowMeans(mat[, groups$WT_Control,    drop = FALSE], na.rm = TRUE),
    WT_LPS        = rowMeans(mat[, groups$WT_LPS,        drop = FALSE], na.rm = TRUE),
    Sp3KO_Control = rowMeans(mat[, groups$Sp3KO_Control, drop = FALSE], na.rm = TRUE),
    Sp3KO_LPS     = rowMeans(mat[, groups$Sp3KO_LPS,     drop = FALSE], na.rm = TRUE)
  )
  rownames(collapsed) <- rownames(mat)
  collapsed
}

############################
## Safe row-scaling (z-score per gene)
############################
safe_row_scale <- function(mat) {
  t(apply(mat, 1, function(x) {
    mu <- mean(x, na.rm = TRUE)
    sdv <- sd(x, na.rm = TRUE)
    if (is.na(sdv) || sdv == 0) rep(0, length(x)) else (x - mu) / sdv
  }))
}

############################
## Annotation + colors
############################
make_annotation <- function() {
  annotation_col <- data.frame(
    Genotype  = c("WT","WT","Sp3KO","Sp3KO"),
    Condition = c("Control","LPS","Control","LPS"),
    stringsAsFactors = FALSE
  )
  rownames(annotation_col) <- c("WT_Control","WT_LPS","Sp3KO_Control","Sp3KO_LPS")

  ann_colors <- list(
    Genotype  = c(WT = "#66c2a5", Sp3KO = "#fc8d62"),
    Condition = c(Control = "#8da0cb", LPS = "#e78ac3")
  )
  list(annotation_col = annotation_col, ann_colors = ann_colors)
}

############################
## Heatmap color palette
############################
hm_colors <- colorRampPalette(
  c("#313695", "#74add1", "#ffffbf", "#f46d43", "#a50026")
)(200)

############################
## Align rows between MEF and LMC
## gene_list_upper is from irf$x (uppercased)
############################
subset_and_align_rows <- function(mat_list, gene_list_upper) {
  sub_list <- lapply(mat_list, function(m) {
    keep <- intersect(gene_list_upper, rownames(m))
    m_sub <- m[keep, , drop = FALSE]
    m_sub <- m_sub[match(gene_list_upper, rownames(m_sub), nomatch = 0), , drop = FALSE]
    m_sub
  })
  common_genes <- Reduce(intersect, lapply(sub_list, rownames))
  if (length(common_genes) == 0) stop("No overlapping genes between MEF and LMC.")
  ordered_common <- gene_list_upper[gene_list_upper %in% common_genes]
  aligned <- lapply(sub_list, function(m) m[ordered_common, , drop = FALSE])
  aligned
}

############################
## pheatmap wrapper (no row clustering, scaled)
## NOTE: we KEEP the same aesthetics you pasted:
## - show_rownames = TRUE
## - cellheight = 10, cellwidth = 20
## - cluster_rows = FALSE (locked order)
############################
make_heatmap_grob <- function(mat, title_text, ann) {
  desired_order <- c("WT_Control","WT_LPS","Sp3KO_Control","Sp3KO_LPS")
  mat <- mat[, desired_order, drop = FALSE]

  mat_scaled <- safe_row_scale(mat)

  p <- pheatmap(
    mat_scaled,
    cluster_rows = FALSE,
    cluster_cols = FALSE,
    scale = "none",     # already scaled
    color = hm_colors,
    annotation_col = ann$annotation_col,
    annotation_colors = ann$ann_colors,
    show_rownames = TRUE,     # <- keep row labels visible
    show_colnames = TRUE,
    fontsize_row = 6,
    fontsize_col = 9,
    angle_col = 45,
    cellheight = 6,          # <- NOT thinned
    cellwidth  = 20,
    border_color = NA,
    main = title_text,
    treeheight_row = 0,
    treeheight_col = 0,
    silent = TRUE
  )
  p$gtable
}

############################
## Gene list comes from irf$x
############################
get_irf_gene_list_upper <- function(irf) {
  if (!"x" %in% colnames(irf)) {
    stop("The 'irf' object must have a column named 'x' with gene symbols.")
  }
  unique(toupper(irf$x))
}

############################
## MAIN PIPELINE
############################

## 1. Prepare MEF
MEF_mat <- prep_MEF_matrix(countdata_MEF)
MEF_groups <- list(
  WT_Control    = c("P7_4_Ctrl1","P7_4_Ctrl2","P8_4_Ctrl3","P8_4_Ctrl4","P8_4_Ctrl5"),
  WT_LPS        = c("P8_4_Lps3","P8_4_Lps5","P8_4_Lps4","P7_4_Lps1","P7_4_Lps2"),
  Sp3KO_Control = c("P4_5_Ctrl1","P4_5_Ctrl3","P4_5_Ctrl2"),
  Sp3KO_LPS     = c("P4_5_Lps2","P4_5_Lps3","P4_5_Lps1")
)
MEF_collapsed <- collapse_to_4groups(MEF_mat, MEF_groups)

## 2. Prepare LMC
LMC_mat <- prep_LMC_matrix(countdata_LMC)
LMC_groups <- list(
  WT_Control    = c("X685.2_CTRL","X697.3_CTRL","X697.1_CTRL","X468.4_CTRL"),
  WT_LPS        = c("X468.4_LPS","X685.2_LPS","X697.3_LPS","X697.1_LPS"),
  Sp3KO_Control = c("X685.1_CTRL","X697.5_CTRL","X697.2_CTRL","X468.6_CTRL"),
  Sp3KO_LPS     = c("X697.5_LPS","X697.2_LPS","X685.1_LPS","X468.6_LPS")
)
LMC_collapsed <- collapse_to_4groups(LMC_mat, LMC_groups)

## 3. Build gene list from irf$x
irf_genes_upper <- get_irf_gene_list_upper(irf)

## 4. Subset to irf genes and align row order between MEF and LMC
aligned_list <- subset_and_align_rows(
  list(MEF = MEF_collapsed, LMC = LMC_collapsed),
  irf_genes_upper
)

MEF_final <- aligned_list$MEF
LMC_final <- aligned_list$LMC

## 5. Shared annotation
ann <- make_annotation()

## 6. Make grobs for both heatmaps
grob_mef <- make_heatmap_grob(
  MEF_final,
  title_text = "MEF_IRF",
  ann = ann
)

grob_lmc <- make_heatmap_grob(
  LMC_final,
  title_text = "LMC_IRF",
  ann = ann
)

## 7. Draw side-by-side
grid.newpage()
grid.arrange(grob_mef, grob_lmc, ncol = 2)


In [ ]:
nfkb <- read.csv('nfkb_genes_lmc.csv')

In [ ]:
nfkb$X <- NULL

In [ ]:
nfkb

In [ ]:
############################
## Load packages
############################
suppressPackageStartupMessages({
  library(pheatmap)
  library(dplyr)
  library(gridExtra)
  library(grid)
})

############################
## 1. Prepare MEF matrix
############################
prep_MEF_matrix <- function(countdata_MEF) {
  MEF_df <- as.data.frame(countdata_MEF)
  gene_col <- colnames(MEF_df)[1]
  gene_names <- toupper(MEF_df[[gene_col]])
  MEF_df[[gene_col]] <- NULL
  MEF_mat <- as.matrix(MEF_df)
  rownames(MEF_mat) <- gene_names
  MEF_mat
}

############################
## 2. Prepare LMC matrix
############################
prep_LMC_matrix <- function(countdata_LMC) {
  LMC_df <- as.data.frame(countdata_LMC)
  if (!"Gene_name" %in% colnames(LMC_df)) {
    stop("countdata_LMC must contain a 'Gene_name' column.")
  }
  gene_names <- toupper(LMC_df$Gene_name)
  LMC_df$Gene_name <- NULL
  if ("X" %in% colnames(LMC_df)) {
    LMC_df$X <- NULL
  }
  LMC_mat <- as.matrix(LMC_df)
  rownames(LMC_mat) <- gene_names
  LMC_mat
}

############################
## 3. Collapse replicates to 4 biological groups
############################
collapse_to_4groups <- function(mat, groups) {
  stopifnot(all(unlist(groups) %in% colnames(mat)))
  collapsed <- data.frame(
    WT_Control    = rowMeans(mat[, groups$WT_Control,    drop = FALSE], na.rm = TRUE),
    WT_LPS        = rowMeans(mat[, groups$WT_LPS,        drop = FALSE], na.rm = TRUE),
    Sp3KO_Control = rowMeans(mat[, groups$Sp3KO_Control, drop = FALSE], na.rm = TRUE),
    Sp3KO_LPS     = rowMeans(mat[, groups$Sp3KO_LPS,     drop = FALSE], na.rm = TRUE)
  )
  rownames(collapsed) <- rownames(mat)
  collapsed
}

############################
## 4. Safe row-wise scaling (z-score per gene)
############################
safe_row_scale <- function(mat) {
  t(apply(mat, 1, function(x) {
    mu  <- mean(x, na.rm = TRUE)
    sdv <- sd(x,  na.rm = TRUE)
    if (is.na(sdv) || sdv == 0) {
      rep(0, length(x))
    } else {
      (x - mu) / sdv
    }
  }))
}

############################
## 5. Condition annotations + colors
############################
make_annotation <- function() {
  annotation_col <- data.frame(
    Genotype  = c("WT","WT","Sp3KO","Sp3KO"),
    Condition = c("Control","LPS","Control","LPS"),
    stringsAsFactors = FALSE
  )
  rownames(annotation_col) <- c("WT_Control","WT_LPS","Sp3KO_Control","Sp3KO_LPS")

  ann_colors <- list(
    Genotype  = c(WT = "#66c2a5", Sp3KO = "#fc8d62"),
    Condition = c(Control = "#8da0cb", LPS = "#e78ac3")
  )

  list(
    annotation_col = annotation_col,
    ann_colors     = ann_colors
  )
}

############################
## 6. Heatmap palette
############################
hm_colors <- colorRampPalette(
  c("#313695", "#74add1", "#ffffbf", "#f46d43", "#a50026")
)(200)

############################
## 7. Get NFkB list from nfkb$x
############################
get_nfkb_gene_list_upper <- function(nfkb) {
  if (!"x" %in% colnames(nfkb)) {
    stop("nfkb must have a column named 'x'.")
  }
  unique(toupper(nfkb$x))
}

############################
## 8. Compute dynamic range per gene
##    For each gene (row), range = max(mean cols) - min(mean cols)
############################
gene_range <- function(collapsed_mat) {
  apply(collapsed_mat, 1, function(v) {
    max(v, na.rm = TRUE) - min(v, na.rm = TRUE)
  })
}

############################
## 9. Pick top N changing genes across MEF and LMC
##    - keep only genes in nfkb list
##    - compute range in MEF and LMC
##    - take union of top_n from both, ordered by max(range_MEF, range_LMC)
############################
select_top_variable_genes <- function(MEF_collapsed, LMC_collapsed, gene_pool_upper, top_n = 100) {

  # limit to nfkb genes present
  MEF_in <- intersect(gene_pool_upper, rownames(MEF_collapsed))
  LMC_in <- intersect(gene_pool_upper, rownames(LMC_collapsed))

  # compute per-gene range
  rMEF <- gene_range(MEF_collapsed[MEF_in, , drop = FALSE])
  rLMC <- gene_range(LMC_collapsed[LMC_in, , drop = FALSE])

  # sort each by range, take top_n gene names
  top_mef_genes <- names(sort(rMEF, decreasing = TRUE))[1:min(top_n, length(rMEF))]
  top_lmc_genes <- names(sort(rLMC, decreasing = TRUE))[1:min(top_n, length(rLMC))]

  # union, then rank that union by max(range in either dataset)
  combined <- union(top_mef_genes, top_lmc_genes)

  # build a combined score = max(range in MEF, range in LMC) for each gene
  combined_score <- sapply(combined, function(g) {
    s_mef <- if (g %in% names(rMEF)) rMEF[[g]] else 0
    s_lmc <- if (g %in% names(rLMC)) rLMC[[g]] else 0
    max(s_mef, s_lmc)
  })

  # order by score desc, keep first top_n
  ordered <- combined[order(combined_score, decreasing = TRUE)]
  ordered[1:min(top_n, length(ordered))]
}

############################
## 10. Subset and align MEF/LMC to a fixed gene order
############################
subset_and_align_rows_fixed <- function(mat_list, ordered_genes) {
  sub_list <- lapply(mat_list, function(m) {
    keep <- intersect(ordered_genes, rownames(m))
    m_sub <- m[keep, , drop = FALSE]
    # enforce target ordering
    m_sub <- m_sub[match(ordered_genes, rownames(m_sub), nomatch = 0), , drop = FALSE]
    m_sub
  })

  # require genes that are in BOTH after subsetting/ordering
  common_genes <- Reduce(intersect, lapply(sub_list, rownames))
  if (length(common_genes) == 0) {
    stop("After selecting top variable genes, none are shared between MEF and LMC.")
  }

  # final enforced order = the ordered_genes that are common to both
  final_order <- ordered_genes[ordered_genes %in% common_genes]

  aligned <- lapply(sub_list, function(m) {
    m[final_order, , drop = FALSE]
  })

  aligned
}

############################
## 11. Heatmap wrapper: thin rows, hide rownames for clarity
############################
make_heatmap_grob <- function(mat, title_text, ann) {
  desired_order <- c("WT_Control","WT_LPS","Sp3KO_Control","Sp3KO_LPS")
  mat <- mat[, desired_order, drop = FALSE]

  mat_scaled <- safe_row_scale(mat)

  p <- pheatmap(
    mat_scaled,
    cluster_rows = FALSE,
    cluster_cols = FALSE,
    scale = "none",
    color = hm_colors,
    annotation_col = ann$annotation_col,
    annotation_colors = ann$ann_colors,
    show_rownames = FALSE,   # too dense to read 100 labels vertically anyway
    show_colnames = TRUE,
    fontsize_col = 9,
    angle_col = 45,
    cellheight = 4,          # thinner than default but readable for ~100 genes
    cellwidth  = 15,
    border_color = NA,
    main = title_text,
    treeheight_row = 0,
    treeheight_col = 0,
    silent = TRUE
  )

  p$gtable
}

############################
## 12. MAIN PIPELINE
############################

## A. Prepare matrices
MEF_mat <- prep_MEF_matrix(countdata_MEF)
LMC_mat <- prep_LMC_matrix(countdata_LMC)

## B. Define replicate groupings
MEF_groups <- list(
  WT_Control    = c("P7_4_Ctrl1","P7_4_Ctrl2","P8_4_Ctrl3","P8_4_Ctrl4","P8_4_Ctrl5"),
  WT_LPS        = c("P8_4_Lps3","P8_4_Lps5","P8_4_Lps4","P7_4_Lps1","P7_4_Lps2"),
  Sp3KO_Control = c("P4_5_Ctrl1","P4_5_Ctrl3","P4_5_Ctrl2"),
  Sp3KO_LPS     = c("P4_5_Lps2","P4_5_Lps3","P4_5_Lps1")
)
LMC_groups <- list(
  WT_Control    = c("X685.2_CTRL","X697.3_CTRL","X697.1_CTRL","X468.4_CTRL"),
  WT_LPS        = c("X468.4_LPS","X685.2_LPS","X697.3_LPS","X697.1_LPS"),
  Sp3KO_Control = c("X685.1_CTRL","X697.5_CTRL","X697.2_CTRL","X468.6_CTRL"),
  Sp3KO_LPS     = c("X697.5_LPS","X697.2_LPS","X685.1_LPS","X468.6_LPS")
)

## C. Collapse to 4-condition profiles
MEF_collapsed <- collapse_to_4groups(MEF_mat, MEF_groups)
LMC_collapsed <- collapse_to_4groups(LMC_mat, LMC_groups)

## D. Get NFκB gene list (uppercase)
nfkb_genes_upper <- get_nfkb_gene_list_upper(nfkb)

## E. Pick top 100 most changed genes across MEF/LMC
top100_genes <- select_top_variable_genes(
  MEF_collapsed,
  LMC_collapsed,
  nfkb_genes_upper,
  top_n = 100
)

## F. Subset + align both matrices to that final gene order
aligned_list <- subset_and_align_rows_fixed(
  list(MEF = MEF_collapsed, LMC = LMC_collapsed),
  top100_genes
)

MEF_final <- aligned_list$MEF
LMC_final <- aligned_list$LMC

## G. Shared annotation
ann <- make_annotation()

## H. Build grobs
grob_mef <- make_heatmap_grob(
  MEF_final,
  title_text = "MEF (NFκB signature, top 100 most changed)",
  ann = ann
)
grob_lmc <- make_heatmap_grob(
  LMC_final,
  title_text = "LMC (NFκB signature, top 100 most changed)",
  ann = ann
)

## I. Draw side by side
grid.newpage()
grid.arrange(grob_mef, grob_lmc, ncol = 2)


In [ ]:
############################
## Load packages
############################
suppressPackageStartupMessages({
  library(pheatmap)
  library(dplyr)
  library(gridExtra)
  library(grid)
})

############################
## Helper functions
############################

prep_MEF_matrix <- function(countdata_MEF) {
  MEF_df <- as.data.frame(countdata_MEF)
  gene_col <- colnames(MEF_df)[1]
  gene_names <- toupper(MEF_df[[gene_col]])
  MEF_df[[gene_col]] <- NULL
  MEF_mat <- as.matrix(MEF_df)
  rownames(MEF_mat) <- gene_names
  MEF_mat
}

prep_LMC_matrix <- function(countdata_LMC) {
  LMC_df <- as.data.frame(countdata_LMC)
  if (!"Gene_name" %in% colnames(LMC_df)) stop("countdata_LMC must contain a 'Gene_name' column.")
  gene_names <- toupper(LMC_df$Gene_name)
  LMC_df$Gene_name <- NULL
  if ("X" %in% colnames(LMC_df)) LMC_df$X <- NULL
  LMC_mat <- as.matrix(LMC_df)
  rownames(LMC_mat) <- gene_names
  LMC_mat
}

collapse_to_4groups <- function(mat, groups) {
  stopifnot(all(unlist(groups) %in% colnames(mat)))
  collapsed <- data.frame(
    WT_Control    = rowMeans(mat[, groups$WT_Control,    drop = FALSE], na.rm = TRUE),
    WT_LPS        = rowMeans(mat[, groups$WT_LPS,        drop = FALSE], na.rm = TRUE),
    Sp3KO_Control = rowMeans(mat[, groups$Sp3KO_Control, drop = FALSE], na.rm = TRUE),
    Sp3KO_LPS     = rowMeans(mat[, groups$Sp3KO_LPS,     drop = FALSE], na.rm = TRUE)
  )
  rownames(collapsed) <- rownames(mat)
  collapsed
}

safe_row_scale <- function(mat) {
  t(apply(mat, 1, function(x) {
    mu  <- mean(x, na.rm = TRUE)
    sdv <- sd(x,  na.rm = TRUE)
    if (is.na(sdv) || sdv == 0) rep(0, length(x)) else (x - mu) / sdv
  }))
}

make_annotation <- function() {
  annotation_col <- data.frame(
    Genotype  = c("WT","WT","Sp3KO","Sp3KO"),
    Condition = c("Control","LPS","Control","LPS"),
    stringsAsFactors = FALSE
  )
  rownames(annotation_col) <- c("WT_Control","WT_LPS","Sp3KO_Control","Sp3KO_LPS")
  ann_colors <- list(
    Genotype  = c(WT = "#66c2a5", Sp3KO = "#fc8d62"),
    Condition = c(Control = "#8da0cb", LPS = "#e78ac3")
  )
  list(annotation_col = annotation_col, ann_colors = ann_colors)
}

hm_colors <- colorRampPalette(
  c("#313695", "#74add1", "#ffffbf", "#f46d43", "#a50026")
)(200)

get_nfkb_gene_list_upper <- function(nfkb) {
  if (!"x" %in% colnames(nfkb)) stop("nfkb must have a column named 'x'.")
  unique(toupper(nfkb$x))
}

gene_range <- function(collapsed_mat) {
  apply(collapsed_mat, 1, function(v) max(v, na.rm = TRUE) - min(v, na.rm = TRUE))
}

select_top_variable_genes <- function(MEF_collapsed, LMC_collapsed, gene_pool_upper, top_n = 50) {
  MEF_in <- intersect(gene_pool_upper, rownames(MEF_collapsed))
  LMC_in <- intersect(gene_pool_upper, rownames(LMC_collapsed))
  rMEF <- gene_range(MEF_collapsed[MEF_in, , drop = FALSE])
  rLMC <- gene_range(LMC_collapsed[LMC_in, , drop = FALSE])
  top_mef_genes <- names(sort(rMEF, decreasing = TRUE))[1:min(top_n, length(rMEF))]
  top_lmc_genes <- names(sort(rLMC, decreasing = TRUE))[1:min(top_n, length(rLMC))]
  combined <- union(top_mef_genes, top_lmc_genes)
  combined_score <- sapply(combined, function(g) {
    s_mef <- if (g %in% names(rMEF)) rMEF[[g]] else 0
    s_lmc <- if (g %in% names(rLMC)) rLMC[[g]] else 0
    max(s_mef, s_lmc)
  })
  ordered <- combined[order(combined_score, decreasing = TRUE)]
  ordered[1:min(top_n, length(ordered))]
}

subset_and_align_rows_fixed <- function(mat_list, ordered_genes) {
  sub_list <- lapply(mat_list, function(m) {
    keep <- intersect(ordered_genes, rownames(m))
    m_sub <- m[keep, , drop = FALSE]
    m_sub <- m_sub[match(ordered_genes, rownames(m_sub), nomatch = 0), , drop = FALSE]
    m_sub
  })
  common_genes <- Reduce(intersect, lapply(sub_list, rownames))
  final_order <- ordered_genes[ordered_genes %in% common_genes]
  aligned <- lapply(sub_list, function(m) m[final_order, , drop = FALSE])
  aligned
}

make_heatmap_grob <- function(mat, title_text, ann) {
  desired_order <- c("WT_Control","WT_LPS","Sp3KO_Control","Sp3KO_LPS")
  mat <- mat[, desired_order, drop = FALSE]
  mat_scaled <- safe_row_scale(mat)
  p <- pheatmap(
    mat_scaled,
    cluster_rows = FALSE,
    cluster_cols = FALSE,
    scale = "none",
    color = hm_colors,
    annotation_col = ann$annotation_col,
    annotation_colors = ann$ann_colors,
    show_rownames = FALSE,
    show_colnames = TRUE,
    fontsize_col = 9,
    angle_col = 45,
    cellheight = 5,
    cellwidth  = 15,
    border_color = NA,
    main = title_text,
    treeheight_row = 0,
    treeheight_col = 0,
    silent = TRUE
  )
  p$gtable
}

############################
## MAIN PIPELINE
############################

MEF_mat <- prep_MEF_matrix(countdata_MEF)
LMC_mat <- prep_LMC_matrix(countdata_LMC)

MEF_groups <- list(
  WT_Control    = c("P7_4_Ctrl1","P7_4_Ctrl2","P8_4_Ctrl3","P8_4_Ctrl4","P8_4_Ctrl5"),
  WT_LPS        = c("P8_4_Lps3","P8_4_Lps5","P8_4_Lps4","P7_4_Lps1","P7_4_Lps2"),
  Sp3KO_Control = c("P4_5_Ctrl1","P4_5_Ctrl3","P4_5_Ctrl2"),
  Sp3KO_LPS     = c("P4_5_Lps2","P4_5_Lps3","P4_5_Lps1")
)

LMC_groups <- list(
  WT_Control    = c("X685.2_CTRL","X697.3_CTRL","X697.1_CTRL","X468.4_CTRL"),
  WT_LPS        = c("X468.4_LPS","X685.2_LPS","X697.3_LPS","X697.1_LPS"),
  Sp3KO_Control = c("X685.1_CTRL","X697.5_CTRL","X697.2_CTRL","X468.6_CTRL"),
  Sp3KO_LPS     = c("X697.5_LPS","X697.2_LPS","X685.1_LPS","X468.6_LPS")
)

MEF_collapsed <- collapse_to_4groups(MEF_mat, MEF_groups)
LMC_collapsed <- collapse_to_4groups(LMC_mat, LMC_groups)

nfkb_genes_upper <- get_nfkb_gene_list_upper(nfkb)

top50_genes <- select_top_variable_genes(
  MEF_collapsed,
  LMC_collapsed,
  nfkb_genes_upper,
  top_n = 50
)

aligned_list <- subset_and_align_rows_fixed(
  list(MEF = MEF_collapsed, LMC = LMC_collapsed),
  top50_genes
)

MEF_final <- aligned_list$MEF
LMC_final <- aligned_list$LMC

ann <- make_annotation()

grob_mef <- make_heatmap_grob(MEF_final, "MEF", ann)
grob_lmc <- make_heatmap_grob(LMC_final, "LMC", ann)

grid.newpage()
grid.arrange(grob_mef, grob_lmc, ncol = 2)


In [ ]:
############################
## Load packages
############################
suppressPackageStartupMessages({
  library(pheatmap)
  library(dplyr)
  library(gridExtra)
  library(grid)
})

############################
## Helper functions
############################

prep_MEF_matrix <- function(countdata_MEF) {
  MEF_df <- as.data.frame(countdata_MEF)
  gene_col <- colnames(MEF_df)[1]
  gene_names <- toupper(MEF_df[[gene_col]])
  MEF_df[[gene_col]] <- NULL
  MEF_mat <- as.matrix(MEF_df)
  rownames(MEF_mat) <- gene_names
  MEF_mat
}

prep_LMC_matrix <- function(countdata_LMC) {
  LMC_df <- as.data.frame(countdata_LMC)
  if (!"Gene_name" %in% colnames(LMC_df)) stop("countdata_LMC must contain a 'Gene_name' column.")
  gene_names <- toupper(LMC_df$Gene_name)
  LMC_df$Gene_name <- NULL
  if ("X" %in% colnames(LMC_df)) LMC_df$X <- NULL
  LMC_mat <- as.matrix(LMC_df)
  rownames(LMC_mat) <- gene_names
  LMC_mat
}

collapse_to_4groups <- function(mat, groups) {
  stopifnot(all(unlist(groups) %in% colnames(mat)))
  collapsed <- data.frame(
    WT_Control    = rowMeans(mat[, groups$WT_Control,    drop = FALSE], na.rm = TRUE),
    WT_LPS        = rowMeans(mat[, groups$WT_LPS,        drop = FALSE], na.rm = TRUE),
    Sp3KO_Control = rowMeans(mat[, groups$Sp3KO_Control, drop = FALSE], na.rm = TRUE),
    Sp3KO_LPS     = rowMeans(mat[, groups$Sp3KO_LPS,     drop = FALSE], na.rm = TRUE)
  )
  rownames(collapsed) <- rownames(mat)
  collapsed
}

safe_row_scale <- function(mat) {
  t(apply(mat, 1, function(x) {
    mu  <- mean(x, na.rm = TRUE)
    sdv <- sd(x,  na.rm = TRUE)
    if (is.na(sdv) || sdv == 0) rep(0, length(x)) else (x - mu) / sdv
  }))
}

make_annotation <- function() {
  annotation_col <- data.frame(
    Genotype  = c("WT","WT","Sp3KO","Sp3KO"),
    Condition = c("Control","LPS","Control","LPS"),
    stringsAsFactors = FALSE
  )
  rownames(annotation_col) <- c("WT_Control","WT_LPS","Sp3KO_Control","Sp3KO_LPS")
  ann_colors <- list(
    Genotype  = c(WT = "#66c2a5", Sp3KO = "#fc8d62"),
    Condition = c(Control = "#8da0cb", LPS = "#e78ac3")
  )
  list(annotation_col = annotation_col, ann_colors = ann_colors)
}

hm_colors <- colorRampPalette(
  c("#313695", "#74add1", "#ffffbf", "#f46d43", "#a50026")
)(200)

get_nfkb_gene_list_upper <- function(nfkb) {
  if (!"x" %in% colnames(nfkb)) stop("nfkb must have a column named 'x'.")
  unique(toupper(nfkb$x))
}

gene_range <- function(collapsed_mat) {
  apply(collapsed_mat, 1, function(v) max(v, na.rm = TRUE) - min(v, na.rm = TRUE))
}

select_top_variable_genes <- function(MEF_collapsed, LMC_collapsed, gene_pool_upper, top_n = 50) {
  MEF_in <- intersect(gene_pool_upper, rownames(MEF_collapsed))
  LMC_in <- intersect(gene_pool_upper, rownames(LMC_collapsed))
  rMEF <- gene_range(MEF_collapsed[MEF_in, , drop = FALSE])
  rLMC <- gene_range(LMC_collapsed[LMC_in, , drop = FALSE])
  top_mef_genes <- names(sort(rMEF, decreasing = TRUE))[1:min(top_n, length(rMEF))]
  top_lmc_genes <- names(sort(rLMC, decreasing = TRUE))[1:min(top_n, length(rLMC))]
  combined <- union(top_mef_genes, top_lmc_genes)
  combined_score <- sapply(combined, function(g) {
    s_mef <- if (g %in% names(rMEF)) rMEF[[g]] else 0
    s_lmc <- if (g %in% names(rLMC)) rLMC[[g]] else 0
    max(s_mef, s_lmc)
  })
  ordered <- combined[order(combined_score, decreasing = TRUE)]
  ordered[1:min(top_n, length(ordered))]
}

subset_and_align_rows_fixed <- function(mat_list, ordered_genes) {
  sub_list <- lapply(mat_list, function(m) {
    keep <- intersect(ordered_genes, rownames(m))
    m_sub <- m[keep, , drop = FALSE]
    m_sub <- m_sub[match(ordered_genes, rownames(m_sub), nomatch = 0), , drop = FALSE]
    m_sub
  })
  common_genes <- Reduce(intersect, lapply(sub_list, rownames))
  final_order <- ordered_genes[ordered_genes %in% common_genes]
  aligned <- lapply(sub_list, function(m) m[final_order, , drop = FALSE])
  aligned
}

make_heatmap_grob <- function(mat, title_text, ann) {
  desired_order <- c("WT_Control","WT_LPS","Sp3KO_Control","Sp3KO_LPS")
  mat <- mat[, desired_order, drop = FALSE]
  mat_scaled <- safe_row_scale(mat)
  p <- pheatmap(
    mat_scaled,
    cluster_rows = FALSE,
    cluster_cols = FALSE,
    scale = "none",
    color = hm_colors,
    annotation_col = ann$annotation_col,
    annotation_colors = ann$ann_colors,
    show_rownames = TRUE,     # <— row labels restored
    show_colnames = TRUE,
    fontsize_row = 6.5,       # smaller font to fit 50 genes
    fontsize_col = 9,
    angle_col = 45,
    cellheight = 6,           # compact but readable
    cellwidth  = 15,
    border_color = NA,
    main = title_text,
    treeheight_row = 0,
    treeheight_col = 0,
    silent = TRUE
  )
  p$gtable
}

############################
## MAIN PIPELINE
############################

MEF_mat <- prep_MEF_matrix(countdata_MEF)
LMC_mat <- prep_LMC_matrix(countdata_LMC)

MEF_groups <- list(
  WT_Control    = c("P7_4_Ctrl1","P7_4_Ctrl2","P8_4_Ctrl3","P8_4_Ctrl4","P8_4_Ctrl5"),
  WT_LPS        = c("P8_4_Lps3","P8_4_Lps5","P8_4_Lps4","P7_4_Lps1","P7_4_Lps2"),
  Sp3KO_Control = c("P4_5_Ctrl1","P4_5_Ctrl3","P4_5_Ctrl2"),
  Sp3KO_LPS     = c("P4_5_Lps2","P4_5_Lps3","P4_5_Lps1")
)

LMC_groups <- list(
  WT_Control    = c("X685.2_CTRL","X697.3_CTRL","X697.1_CTRL","X468.4_CTRL"),
  WT_LPS        = c("X468.4_LPS","X685.2_LPS","X697.3_LPS","X697.1_LPS"),
  Sp3KO_Control = c("X685.1_CTRL","X697.5_CTRL","X697.2_CTRL","X468.6_CTRL"),
  Sp3KO_LPS     = c("X697.5_LPS","X697.2_LPS","X685.1_LPS","X468.6_LPS")
)

MEF_collapsed <- collapse_to_4groups(MEF_mat, MEF_groups)
LMC_collapsed <- collapse_to_4groups(LMC_mat, LMC_groups)

nfkb_genes_upper <- get_nfkb_gene_list_upper(nfkb)

top50_genes <- select_top_variable_genes(
  MEF_collapsed,
  LMC_collapsed,
  nfkb_genes_upper,
  top_n = 50
)

aligned_list <- subset_and_align_rows_fixed(
  list(MEF = MEF_collapsed, LMC = LMC_collapsed),
  top50_genes
)

MEF_final <- aligned_list$MEF
LMC_final <- aligned_list$LMC

ann <- make_annotation()

grob_mef <- make_heatmap_grob(MEF_final, "MEF ", ann)
grob_lmc <- make_heatmap_grob(LMC_final, "LMC", ann)

grid.newpage()
grid.arrange(grob_mef, grob_lmc, ncol = 2)
